# 先頭の数文を要約とする (Lead-3) ベースラインと比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_summarization/lead3_baseline.ipynb)

NLP クックブック 24-03 ／ 要約と平易化

凝った手法と比べる前に、「先頭の 3 文を取るだけ」という最も単純な方法でどこまでいけるかを確かめたい。

In [1]:
!pip install -q rouge-score fugashi unidic-lite networkx

In [2]:
import io, re, tarfile, urllib.request
import pandas as pd

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[。！？])|\n+", text) if s and len(s.strip()) > 5]

df = load_livedoor()
df = df[df["label"].isin(["topic-news", "sports-watch", "it-life-hack"])].sample(150, random_state=0).reset_index(drop=True)
df["sents"] = df["text"].map(split_sentences)
df = df[df["sents"].map(len) >= 8].reset_index(drop=True)
print(len(df), "記事 / 平均", round(df["sents"].map(len).mean(), 1), "文")

135 記事 / 平均 23.5 文


In [3]:
import random
import networkx as nx
import numpy as np
from fugashi import Tagger
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer

tagger = Tagger()
content = lambda s: [w.surface for w in tagger(s) if w.feature.pos1 in ("名詞", "動詞", "形容詞")]

class Tok:
    def tokenize(self, text):
        return [w.surface for w in tagger(text)]

def lead(sents, n=3):
    return sents[:n]

def random_pick(sents, n=3, seed=0):
    return [sents[i] for i in sorted(random.Random(seed).sample(range(len(sents)), n))]

def lexrank(sents, n=3):
    X = TfidfVectorizer(tokenizer=content, token_pattern=None, lowercase=False).fit_transform(sents)
    sim = (X @ X.T).toarray(); np.fill_diagonal(sim, 0)
    scores = nx.pagerank(nx.from_numpy_array(sim), weight="weight")
    return [sents[i] for i in sorted(sorted(scores, key=scores.get, reverse=True)[:n])]

# 人手の要約がないので、記事のタイトルを「参照要約」の代わりに使う (タイトルの語をどれだけ拾えたか = ROUGE の再現率)
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2"], tokenizer=Tok())
rows = []
for name, method in [("ランダムな 3 文", random_pick), ("Lead-3 (先頭 3 文)", lead), ("LexRank", lexrank)]:
    scores = [scorer.score(title, "".join(method(sents))) for title, sents in zip(df["title"], df["sents"])]
    rows.append({"手法": name, "ROUGE-1 再現率": np.mean([s["rouge1"].recall for s in scores]), "ROUGE-2 再現率": np.mean([s["rouge2"].recall for s in scores]),
                 "平均文字数": np.mean([len("".join(method(s))) for s in df["sents"]])})
pd.DataFrame(rows).round(3)

,手法,ROUGE-1 再現率,ROUGE-2 再現率,平均文字数
0,ランダムな 3 文,0.443,0.130,130.459
1,Lead-3 (先頭 3 文),0.544,0.192,181.467
2,LexRank,0.524,0.199,182.941


In [4]:
i = 0
print("タイトル:", df["title"][i])
print("Lead-3 :", " ".join(lead(df["sents"][i])))
print("LexRank:", " ".join(lexrank(df["sents"][i])))

タイトル: 【韓国ニュース】韓国旗踏みつけ動画に批判殺到　韓国ユーザー「米国相手にはできないくせに」 
Lead-3 : 26日、韓国旗に似たマットを踏みつける動画が動画サイトに投稿され、韓国でも波紋を呼んでいる。 動画は、「日韓断交共闘委員会・関西」なる市民団体が主催し、日韓自由貿易協定や日韓防衛協定阻止し、日韓国交断絶を呼びかけるデモの一部を撮影したもので、韓国旗に似たマットを参加者たちが次々と踏みつける様子が収められている。 同事件は韓国紙「国民日報」でも報道され、記事のコメント欄には「国交断絶するなら天皇陛下を返還しろ」「米国相手にはこんなことできないくせに」「北朝鮮と似た行動だ」「他の日本人も恥ずかしいだろう」など、デモ主催者に対する批判が殺到している。
LexRank: 26日、韓国旗に似たマットを踏みつける動画が動画サイトに投稿され、韓国でも波紋を呼んでいる。 動画は、「日韓断交共闘委員会・関西」なる市民団体が主催し、日韓自由貿易協定や日韓防衛協定阻止し、日韓国交断絶を呼びかけるデモの一部を撮影したもので、韓国旗に似たマットを参加者たちが次々と踏みつける様子が収められている。 同事件は韓国紙「国民日報」でも報道され、記事のコメント欄には「国交断絶するなら天皇陛下を返還しろ」「米国相手にはこんなことできないくせに」「北朝鮮と似た行動だ」「他の日本人も恥ずかしいだろう」など、デモ主催者に対する批判が殺到している。


- ニュース記事は重要なことを最初に書く (逆ピラミッド型) ので、Lead-3 は非常に強い。凝った手法が Lead-3 に勝てないことは珍しくない。
- 比べている LexRank と、日本語のトークナイザを渡した ROUGE の使い方は、この章の「重要な文を選んで抽出型要約を作る (LexRank)」と「要約を ROUGE で評価する (日本語対応)」で説明した。
- 新しい要約手法を評価するときは、必ず Lead-3 とランダムを並べて報告する。
- ここではタイトルを参照の代わりにした簡易な評価。本来は人手で書いた参照要約 (日本語なら livedoor の 3 行要約、XL-Sum の日本語部分、WikiLingua など) を使う。
- 小説、論文、議事録のように、重要な内容が先頭にあるとは限らない文書では、Lead の優位は小さくなる。